# ScamRoad — Pothole & Road Damage Detector Training (Kaggle GPU)

**Architecture**: Ultralytics YOLOv8n / YOLOv8s  
**Hardware**: Kaggle Free GPU (Tesla T4 or P100 with 16GB VRAM)  
**Target Deployment**: Mobile (Android / OPPO K13) and Edge SBCs (Raspberry Pi / Jetson)  
**Single Source Taxonomy**: `pothole`, `crack_longitudinal`, `crack_transverse`, `crack_alligator`  

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics pyyaml

In [ ]:
# 3. Prepare ScamRoad Dataset Configuration
from pathlib import Path

import yaml

workspace = Path("/kaggle/working/scamroad_ml")
workspace.mkdir(parents=True, exist_ok=True)

data_cfg = {
    "path": "/kaggle/working/data",
    "train": "splits/train.txt",
    "val": "splits/val.txt",
    "test": "splits/test.txt",
    "nc": 4,
    "names": {
        0: "pothole",
        1: "crack_longitudinal",
        2: "crack_transverse",
        3: "crack_alligator"
    }
}

cfg_file = workspace / "dataset.yaml"
with open(cfg_file, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f)

print(f"Dataset configuration written to {cfg_file}")

In [ ]:
# 4. Launch YOLOv8n Training
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(cfg_file),
    epochs=100,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/scamroad_ml/runs",
    name="pothole_yolov8n",
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3.0,
    patience=25,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.10,
    scale=0.50,
    mosaic=1.0,
    mixup=0.10,
    save=True,
    plots=True
)

print("Training completed successfully!")

In [ ]:
# 5. Run Evaluation on Validation Split
val_metrics = model.val(
    data=str(cfg_file),
    split="val",
    imgsz=640,
    conf=0.25,
    iou=0.45
)

print("=== Validation Metrics ===")
print(f"mAP@50:    {val_metrics.box.map50:.4f}")
print(f"mAP@50-95: {val_metrics.box.map:.4f}")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")

In [ ]:
# 6. Export to ONNX and TFLite for Edge Deployment (Phase 4 Ready)
best_weights = "/kaggle/working/scamroad_ml/runs/pothole_yolov8n/weights/best.pt"
export_model = YOLO(best_weights)

# Export ONNX
export_model.export(format="onnx", imgsz=640, opset=17)
print("ONNX export completed!")